# Final Project: KLUE-BERT Korean Domain and Formality Classfication <br>

Eungyeol(Erin) Bae <br>
eungyeol.bae.th@dartmouth.edu <br>
June 6 2023 <br>


In this notebook, we will use pre-trained deep learning model to process some text. We will then use the output of that model to classify the text. The text is a list of sentences from Academic, General, Colloquial, and Literary texts. The project will classify the text into 4 domains listed above, and 3 formalities.

## Models: Text Classification <br>
KLUE BERT

## Dataset <br>
Academic: https://www.kci.go.kr/kciportal/main.kci <br>
General: https://dumps.wikimedia.org/wikidatawiki/entities/ <br>
Literary: https://sf.jikji.org/book/index.html https://gongu.copyright.or.kr <br>
Colloquial: https://huggingface.co/datasets/NX2411/AIhub-korean-speech-data-large <br>


## Installing the transformers library <br>
Let's start by installing the huggingface transformers library so we can load our deep learning NLP model.

In [3]:
!pip install transformers
!pip install datasets
!pip install -U --no-cache-dir gdown --pre
!pip install pyarrow
!pip install nltk
!pip install torch


In [4]:
import numpy as np
import pandas as pd
import gdown
import pyarrow as pa
import sys
import nltk
import nltk
nltk.download('punkt')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_score, accuracy_score, f1_score, recall_score, classification_report, multilabel_confusion_matrix
import torch
import transformers as ppb
import warnings
warnings.filterwarnings('ignore')
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import AutoModel, AutoTokenizer
import os
import string
import sys
import re
from nltk.tokenize import sent_tokenize
from statistics import mode
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/erinbae/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/erinbae/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Importing the dataset
We'll use pandas to read the dataset and load it into a dataframe.

In [5]:
# Download files
# Academic
urlAca = "https://drive.google.com/uc?export=download&id=14AI5WYQDDluJrYJk55caORCPNOzad-uh"
outputAca = 'acadmic.txt'
gdown.download(urlAca, outputAca, quiet=False)

#General
urlGen = "https://drive.google.com/uc?export=download&id=12ggkc_XAVIIBhXPA47-4G8k3I-z2XPrN"
outputGen = 'general.txt'
gdown.download(urlGen, outputGen, quiet=False)

#Creative
urlCre = "https://drive.google.com/uc?export=download&id=1Krd2k_LjblTUrZiEjYMbdQuMdOAaJ8gL"
outputCre = 'creative.txt'
gdown.download(urlCre, outputCre, quiet=False)

#Speech
# Initialize the FileSource object
dataset = load_dataset("NX2411/AIhub-korean-speech-data", split="train")
# Convert the dataset to a pandas DataFrame
Spe = pd.DataFrame(dataset['texts'])
Spe.head()


Downloading...
From: https://drive.google.com/uc?export=download&id=14AI5WYQDDluJrYJk55caORCPNOzad-uh
To: /Users/erinbae/Downloads/cosc72/Model/acadmic.txt
100%|██████████| 2.65M/2.65M [00:00<00:00, 8.32MB/s]
Downloading...
From: https://drive.google.com/uc?export=download&id=12ggkc_XAVIIBhXPA47-4G8k3I-z2XPrN
To: /Users/erinbae/Downloads/cosc72/Model/general.txt
100%|██████████| 4.66M/4.66M [00:05<00:00, 838kB/s] 
Downloading...
From: https://drive.google.com/uc?export=download&id=1Krd2k_LjblTUrZiEjYMbdQuMdOAaJ8gL
To: /Users/erinbae/Downloads/cosc72/Model/creative.txt
100%|██████████| 5.74M/5.74M [00:00<00:00, 13.8MB/s]


,0
0,내 딸에게 남자 친구가 생겼어 근데 좀 슬퍼
1,담임 선생님 계실 땐 친한 척 하면서 선생님이 없을 때 날 못살게 구는 애들 때문에...
2,내 자녀가 모 대학에 합격했어 이렇게 건강하게 자라줘서 너무 고마운거 알지 그러고 ...
3,나를 좋아한다고 했던 사람이 갑자기 쌀쌀맞게 굴어
4,나의 특기와 장점에 대해 많이 고민한 결과 진로에 대해 확신이 생겼어


In [6]:
Aca = pd.read_csv('acadmic.txt', delimiter = '\t')
Aca.columns = ['Text']
Aca.dropna(inplace=True)
Aca['Domain'] = 0

Gen = pd.read_csv('general.txt', delimiter = '\t')
Gen.columns = ['Text']
Gen.dropna(inplace=True)
Gen['Domain'] = 1

Cre = pd.read_csv('creative.txt', delimiter = '\t')
Cre.columns = ['Text']
Cre.dropna(inplace=True)
Cre['Domain'] = 3


Spe.dropna(inplace=True)
Spe.columns = ['Text']
Spe['Domain'] = 2

df = pd.concat([Aca, Gen, Spe, Cre])
df.columns = ['Text', 'Domain']

#df.dropna(inplace=True)
#df.dropna(subset=['Text'], inplace=True)


#df.head(n=50)
#Gen.head(n=20)
#Cre.head(n=20)
#Aca.head(n=20)
#Spe.head(n=20)

For performance reasons, we'll only use 2,000 sentences from the dataset

In [7]:
batch_1 = df.groupby('Domain').sample(n=4500, random_state = 100)
batch_1['Domain'].value_counts()

Domain
0    4500
1    4500
2    4500
3    4500
Name: count, dtype: int64

## Loading the Pre-trained BERT model
Let's now load a pre-trained BERT model.

In [8]:
"""
# For DistilBERT:
#model_class, tokenizer_class, pretrained_weights = (ppb.DistillBertModel, ppb.DistillBertTokenizer, 'distilbert-base-uncased')

## Want BERT instead of distilBERT? Uncomment the following line:
model_class, tokenizer_class, pretrained_weights = (ppb.BertModel, ppb.BertTokenizer, 'bert-base-uncased')

# Load pretrained model/tokenizer
tokenizer = tokenizer_class.from_pretrained(pretrained_weights)
model = model_class.from_pretrained(pretrained_weights)
"""
model = AutoModel.from_pretrained("klue/bert-base")
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")


Right now, the variable `model` holds a pretrained distilBERT model -- a version of BERT that is smaller, but much faster and requiring a lot less memory.

## Model #1: Preparing the Dataset
Before we can hand our sentences to BERT, we need to so some minimal processing to put them in the format it requires.

### Tokenization
Our first step is to tokenize the sentences -- break them up into word and subwords in the format BERT is comfortable with.

In [9]:
tokenized = batch_1["Text"].apply((lambda x: tokenizer.encode(x, add_special_tokens=True, max_length=512, truncation=True)[:512]))

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-tokenization-2-token-ids.png" />

### Padding
After tokenization, `tokenized` is a list of sentences -- each sentences is represented as a list of tokens. We want BERT to process our examples all at once (as one batch). It's just faster that way. For that reason, we need to pad all lists to the same size, so we can represent the input as one 2-d array, rather than a list of lists (of different lengths).

In [10]:
max_len = 0
for i in tokenized.values:
    if len(i) > max_len:
        max_len = len(i)

padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])

Our dataset is now in the `padded` variable, we can view its dimensions below:

In [11]:
np.array(padded).shape

(18000, 512)

### Masking
If we directly send `padded` to BERT, that would slightly confuse it. We need to create another variable to tell it to ignore (mask) the padding we've added when it's processing its input. That's what attention_mask is:

In [13]:
attention_mask = np.where(padded != 0, 1, 0)
attention_mask.shape

(18000, 512)

## Model #1: And Now, Deep Learning!
Now that we have our model and inputs ready, let's run our model!

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-tutorial-sentence-embedding.png" />

The `model()` function runs our sentences through BERT. The results of the processing will be returned into `last_hidden_states`.

In [ ]:
BATCH_SIZE = 32  # You can try 16, 32, or 64 depending on your system
features = []

for i in range(0, len(padded), BATCH_SIZE):
    batch_input_ids = torch.tensor(padded[i:i+BATCH_SIZE])
    batch_attention_mask = torch.tensor(attention_mask[i:i+BATCH_SIZE])

    with torch.no_grad():
        outputs = model(batch_input_ids, attention_mask=batch_attention_mask)

    # Extract [CLS] token embedding
    cls_embeddings = outputs[0][:, 0, :].numpy()
    features.extend(cls_embeddings)


In [17]:
# With a CPU, it takes about 4 minutes
# 7 min for 500 sentence sample


input_ids = torch.tensor(padded)
attention_mask = torch.tensor(attention_mask)

with torch.no_grad():
    last_hidden_states = model(input_ids, attention_mask=attention_mask)

: 

Let's slice only the part of the output that we need. That is the output corresponding the first token of each sentence. The way BERT does sentence classification, is that it adds a token called `[CLS]` (for classification) at the beginning of every sentence. The output corresponding to that token can be thought of as an embedding for the entire sentence.

<img src="https://jalammar.github.io/images/distilBERT/bert-output-tensor-selection.png" />

We'll save those in the `features` variable, as they'll serve as the features to our logitics regression model.

In [ ]:
features = last_hidden_states[0][:,0,:].numpy()

The labels indicating which sentence is positive and negative now go into the `labels` variable

In [ ]:
features = np.array(features)
labels = batch_1['Domain']

## Model #2: Train/Test Split
Let's now split our datset into a training set and testing set (even though we're using 2,000 sentences from the SST2 training set).

In [ ]:
train_features, test_features, train_labels, test_labels = train_test_split(features, labels, shuffle=True)

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-train-test-split-sentence-embedding.png" />

### [Bonus] Grid Search for Parameters
We can dive into Logistic regression directly with the Scikit Learn default parameters, but sometimes it's worth searching for the best value of the C parameter, which determines regularization strength.

In [ ]:
parameters = {'C': np.linspace(0.0001, 100, 20)}
grid_search = GridSearchCV(LogisticRegression(), parameters)
grid_search.fit(train_features, train_labels)

print('best parameters: ', grid_search.best_params_)
print('best scores: ', grid_search.best_score_)

best parameters:  {'C': np.float64(5.263252631578947)}
best scores:  0.9559999999999998


We now train the LogisticRegression model. If you've chosen to do the gridsearch, you can plug the value of C into the model declaration (e.g. `LogisticRegression(C=5.2)`).

In [ ]:
lr_clf = LogisticRegression(C=5.263252631578947)
lr_clf.fit(train_features, train_labels)

LogisticRegression(C=5.263252631578947)

<img src="https://jalammar.github.io/images/distilBERT/bert-training-logistic-regression.png" />

## Evaluating Model #2
So how well does our model do in classifying sentences? One way is to check the accuracy against the testing dataset:

In [ ]:
score = lr_clf.score(test_features, test_labels)
print("Accuracy:", score)

# Predict labels for the test features
test_predictions = lr_clf.predict(test_features)

print('Classification Matrix: \n', classification_report(test_labels, test_predictions, digits=3))
print('Multilabel Confusion Matrix: \n', multilabel_confusion_matrix(test_labels, test_predictions))

Accuracy: 0.952
Classification Matrix: 
               precision    recall  f1-score   support

           0      0.970     0.985     0.977        66
           1      0.877     0.943     0.909        53
           2      1.000     1.000     1.000        74
           3      0.942     0.860     0.899        57

    accuracy                          0.952       250
   macro avg      0.947     0.947     0.946       250
weighted avg      0.953     0.952     0.952       250

Multilabel Confusion Matrix: 
 [[[182   2]
  [  1  65]]

 [[190   7]
  [  3  50]]

 [[176   0]
  [  0  74]]

 [[190   3]
  [  8  49]]]


How good is this score? What can we compare it against? Let's first look at a dummy classifier:

In [ ]:
from sklearn.dummy import DummyClassifier
clf = DummyClassifier()

scores = cross_val_score(clf, train_features, train_labels)
print("Dummy classifier score: %0.3f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))

Dummy classifier score: 0.263 (+/- 0.01)


So our model clearly does better than a dummy classifier. But how does it compare against the best models?

## Proper SST2 scores
For reference, the [highest accuracy score](http://nlpprogress.com/english/sentiment_analysis.html) for this dataset is currently **96.8**. DistilBERT can be trained to improve its score on this task – a process called **fine-tuning** which updates BERT’s weights to make it achieve a better performance in this sentence classification task (which we can call the downstream task). The fine-tuned DistilBERT turns out to achieve an accuracy score of **90.7**. The full size BERT model achieves **94.9**.



And that’s it! That’s a good first contact with BERT. The next step would be to head over to the documentation and try your hand at [fine-tuning](https://huggingface.co/transformers/examples.html#glue). You can also go back and switch from distilBERT to BERT and see how that works.

This function takes in string input and classifies which formality the sentence belongs to, between Formal Honorific, Semiformal Honorific, and Non-honorific.

In [ ]:
def honorifics(input):
  reFormal = r"(니다[.!]|니까[.?]*)"
  groupFormal = re.search(reFormal,input,re.IGNORECASE)
  reSemi = r"(요[.!?]*)"
  groupSemi = re.search(reSemi,input,re.IGNORECASE)
  if (groupFormal != None):
    return 'Formal Honorific'
  if (groupSemi != None):
    return 'Casual Honorific'
  else:
    return 'Non-honorific'

In [ ]:
sent = [honorifics("안녕."), honorifics("안녕하세요?"),honorifics("안녕하십니까!")]
print(sent)

['Non-honorific', 'Casual Honorific', 'Formal Honorific']


This function takes in string input and classifies which domain the text belongs to, between Academic, General, Colloquial, and Literary.

In [ ]:
def domain(input):
  new_input_ids = torch.tensor(tokenizer.encode(input, add_special_tokens=True)).unsqueeze(0)
  new_outputs = model(new_input_ids)
  new_last_hidden_states = [new_outputs[0].detach().numpy()[0][0]]
  prob= lr_clf.predict_proba(new_last_hidden_states)
  index = np.argmax(prob)
  if index == 0:
    genre = 'Academic'
  elif index == 1:
    genre = 'General'
  elif index == 2:
    genre = 'Colloquial'
  elif index == 3:
    genre = 'Literary'
  print("Input:\n", input)
  print("Probability:\n", prob)
  return genre

This function combines the two classifiers into one function.

In [ ]:
def classifier(input):
    sentences = sent_tokenize(input)
    honor = []
    for x in sentences:
      honor.append(honorifics(x))
    freq = mode(honor)
    print("Category:\n", ['Domain: ' + domain(input), 'Audience: ' + freq], '\n')

Example 1: "In the early morning, my dad shouted, "Our time is already gone."" <br> <br>
Example 2: "The Crystal Palace is basically a rectangular hall with a wide, flat roof, and most of the building is flat roofed, except for the central arched transept (51 m high and 22 m wide)." <br> <br>
Example 3: "I'm sad because of the kids who pretend to be close when the homeroom teacher is there and make me miserable when the teacher is not there." <br> <br>
Example 4: "The core competency model for early childhood teachers consisted of eight core competencies, 23 sub-competencies, and 92 sub-content, and eight core competencies of early childhood teachers were developed as teacher personality and professional development, play support, infant growth and development, educational environment and curriculum, family and community, kindergarten operation and class management, and cultural literacy." <br> <br>

In [ ]:
#Sentences
classifier("새벽에 아버지는 이제 우리들 시대는 이미 갔다라며 고래 고래 소리를 질렀어요.")
classifier("수정궁은 기본적으로 넓고 평평한 지붕의 직사각형 홀이며, 중앙의 아치형 트랜셉트(높이 51m, 너비 22m인 부분)을 제외하고는 건물의 대부분은 평면 지붕입니다.")
classifier("담임 선생님 계실 땐 친한 척 하면서 선생님이 없을 때 날 못살게 구는 애들 때문에 서러워.")
classifier("개발된 유아교사 핵심역량모델은 핵심역량 8개, 하위역량 23개, 하위내용 92개로 구성되었으며, 유아교사의 핵심역량 8개 영역은 교사인성 및 전문성 개발, 유아와의 상호작용, 놀이지원, 유아의 성장과 발달, 교육환경과 교육과정, 가족과 지역사회, 유치원운영 및 학급관리, 문화소양으로 개발되었었다.")

Input:
 새벽에 아버지는 이제 우리들 시대는 이미 갔다라며 고래 고래 소리를 질렀어요.
Probability:
 [[3.11312558e-09 8.07478721e-07 1.52908749e-04 9.99846281e-01]]
Category:
 ['Domain: Literary', 'Audience: Casual Honorific'] 

Input:
 수정궁은 기본적으로 넓고 평평한 지붕의 직사각형 홀이며, 중앙의 아치형 트랜셉트(높이 51m, 너비 22m인 부분)을 제외하고는 건물의 대부분은 평면 지붕입니다.
Probability:
 [[1.19482904e-07 9.99999792e-01 7.28054211e-09 8.12949047e-08]]
Category:
 ['Domain: General', 'Audience: Formal Honorific'] 

Input:
 담임 선생님 계실 땐 친한 척 하면서 선생님이 없을 때 날 못살게 구는 애들 때문에 서러워.
Probability:
 [[7.68876877e-09 4.04614094e-07 9.99999048e-01 5.39610131e-07]]
Category:
 ['Domain: Colloquial', 'Audience: Non-honorific'] 

Input:
 개발된 유아교사 핵심역량모델은 핵심역량 8개, 하위역량 23개, 하위내용 92개로 구성되었으며, 유아교사의 핵심역량 8개 영역은 교사인성 및 전문성 개발, 유아와의 상호작용, 놀이지원, 유아의 성장과 발달, 교육환경과 교육과정, 가족과 지역사회, 유치원운영 및 학급관리, 문화소양으로 개발되었었다.
Probability:
 [[9.99967246e-01 3.27533510e-05 9.37355611e-10 1.37134106e-10]]
Category:
 ['Domain: Academic', 'Audience: Non-honorific'] 



Example 1: "After that, one person went to his hometown, Hangok-ri, and one person was a correspondent of the Rural Business Department of the Christian Youth Association, but after going down to the corner of Cheongseokgol, which is completely isolated from all cultural facilities, there was no opportunity to meet each other. There was no time and travel expenses to look for, and there was also a covenant not to meet until the foundation of the project was established to some extent. However, instead, a letter with two or three Samjeon stamps was given to the king once a week and once every ten days without skipping. The content of the letter was not a whisper of the sweet love that is common between young men and women, but purely a business report, an exchange of views, or a real-life struggle. Even if they sat with their eyes closed, they wrote down the situation of Hangok-ri and Cheongseokgol, how they were doing, and even read a book and felt something in their heads." <br> <br>
Example 2: "For this purpose, SVM, a machine learning method for organizing datasets and classifying them into three grades through image data and augmentation techniques by photographing cucumbers grown directly by farmers in the same background, and CNN and VGGNet, a deep learning method, were used. In addition, this study developed an algorithm that produces better classification performance by changing structures, loss and activation functions, and hyper-parameters such as learning rates to automatically learn important patterns and rules from large-scale data and make decisions and predictions. In addition, through experiments, it was confirmed that the proposed algorithm distinguishes cucumbers by grade using image data obtained from agricultural sites." <br> <br>
Example 3: "There are times when I sing or dance because I think I should stand out in my self-introduction. This isn't good either. We usually introduce ourselves at the beginning of the interview. At this point, the interviewer is disappointed, even if the answer is normal, in anticipation of more than that in the next answer. Rather, it is much better to introduce yourself easily and show proper skills in future interviews." <br> <br>
Example 4: "You know, do you have any guilt in your head? You have no skills, no guilt, no feelings for anyone.. Did you think it would be stable if you picked up innocent people, stepped on them thoroughly, and got to the top? If you only knew how to walk on the flowery path, you would have lost it. Don't look, open your eyes and look at what you've done! Is there any reason why you should be confident now? Think if you have a brain and have an accident, how you should behave. You're a lot more stupid than I thought. You're so smart when you trample on others, but you can't do anything when you get caught. If you're going to do it, do it right." <br> <br>

In [ ]:
#Paragraphs
classifier("그 후 한 사람은 고향인 한곡리로, 한 사람은 기독교청년회연합회 농촌사업부의 특파원 격으로 경기 땅이지만 모든 문화시설과는 완전히 격리된 청석골〔靑石洞〕이란 두메 구석으로 내려가서 일터를 잡은 뒤에는 서로 만날 기회가 없었다. 한가히 찾아다닐 시간과 여비까지도 없었거니와, 피차에 사업의 기초가 어느 정도까지 잡히기 전에는 만나지 말자는 언약도 있었던 것이다. 그러나, 그 대신 삼 전짜리 우표가 두 장 혹은 석 장씩 붙은 편지가 일주일에 한 번 열흘에 한 번씩은 거르지 않고 내왕을 하였다. 그 편지의 내용이란, 젊은 남녀간에 흔히 있는 달콤한 사랑을 속삭인 것이 아니라, 순전히 사업 보고요, 의견교환이요, 또는 실제 운동의 고심담이었다. 서로 눈을 감고 앉았어도 한 곡리와 청석골의 형편과 무슨 일을 어떻게 해나가는 것이며, 심지어 틈틈이 무슨 책을 읽고 어떠한 느낌을 받았다는 등 머릿속까지 환하게 들여다보이도록 적어 보냈고 적혀 오고하였다.")
classifier("이를 목적으로 농가에서 직접 재배한 오이를 동일한 배경에서 촬영하여 이미지 데이터와 데이터 증가(augmentation) 기법을 통해 데이터셋을 구성하고 3가지 등급으로 분류하기 위한 기계학습방법인 SVM과 딥러닝 방법인 CNN, VGGNet 등을 사용하였다. 또한 본 연구는 대규모 데이터에서 오이를 기계가 자동으로 중요한 패턴과 규칙을 학습하고 의사결정과 예측 등을 하기 위해 구조나 손실 및 활성화 함수들 그리고 학습비율과 같은 하이퍼 파라미터(hyper-parameter)등을 변경시켜 가며 더 좋은 분류 성능을 내는 알고리즘을 개발하였다. 또한 실험을 통해서 제안된 알고리즘이 농업현장에서 취득한 영상자료를 사용해서 오이를 등급별로 잘 구별하는 것을 확인 할 수 있었다.")
classifier("자기소개에서 눈에 띄어야 한다는 생각에 노래를 부르거나 춤을 추는 경우도 있는데요. 이 방법도 좋지 않습니다. 보통 자기소개는 면접 초반에 진행하는데요. 이 때 눈에 띄게 되면 면접관은 다음 답변에서 그 이상을 기대해 평범하게 답하더라도 실망하게 됩니다. 오히려 자기소개는 평이하게 하고 이후 면접에서 제대로 된 실력을 보여주는 것이 훨씬 좋답니다. ")
classifier("너 말이야, 네 머릿속에 죄책감이라는게 있기는하니? 실력도 죄책감도 누군가를 위한 마음조차도 없으면서.. 죄 없는 사람들 꼬투리 잡아서 철저하게 밟고 정상에 올라가면 안정적일줄 알았어? 꽃길만 걸을 줄 알았다면 네가 헛짚은거야. 어딜 봐,두 눈 뜨고 네가 한 짓을 봐! 이러고도 네가 지금 당당해야 할 이유가 있니? 머리가 있고 사고가 있다면 생각을 해, 네가 어떻게 행동해야 하는지. 생각보다 멍청하네 남 짓밟을때는 그렇게 똑똑하더니 막상 들키니까 아무것도 못하고. 할꺼면 똑바로 해.")

Input:
 그 후 한 사람은 고향인 한곡리로, 한 사람은 기독교청년회연합회 농촌사업부의 특파원 격으로 경기 땅이지만 모든 문화시설과는 완전히 격리된 청석골〔靑石洞〕이란 두메 구석으로 내려가서 일터를 잡은 뒤에는 서로 만날 기회가 없었다. 한가히 찾아다닐 시간과 여비까지도 없었거니와, 피차에 사업의 기초가 어느 정도까지 잡히기 전에는 만나지 말자는 언약도 있었던 것이다. 그러나, 그 대신 삼 전짜리 우표가 두 장 혹은 석 장씩 붙은 편지가 일주일에 한 번 열흘에 한 번씩은 거르지 않고 내왕을 하였다. 그 편지의 내용이란, 젊은 남녀간에 흔히 있는 달콤한 사랑을 속삭인 것이 아니라, 순전히 사업 보고요, 의견교환이요, 또는 실제 운동의 고심담이었다. 서로 눈을 감고 앉았어도 한 곡리와 청석골의 형편과 무슨 일을 어떻게 해나가는 것이며, 심지어 틈틈이 무슨 책을 읽고 어떠한 느낌을 받았다는 등 머릿속까지 환하게 들여다보이도록 적어 보냈고 적혀 오고하였다.
Probability:
 [[1.28819783e-06 2.26971795e-04 2.30342200e-05 9.99748706e-01]]
Category:
 ['Domain: Literary', 'Audience: Non-honorific'] 

Input:
 이를 목적으로 농가에서 직접 재배한 오이를 동일한 배경에서 촬영하여 이미지 데이터와 데이터 증가(augmentation) 기법을 통해 데이터셋을 구성하고 3가지 등급으로 분류하기 위한 기계학습방법인 SVM과 딥러닝 방법인 CNN, VGGNet 등을 사용하였다. 또한 본 연구는 대규모 데이터에서 오이를 기계가 자동으로 중요한 패턴과 규칙을 학습하고 의사결정과 예측 등을 하기 위해 구조나 손실 및 활성화 함수들 그리고 학습비율과 같은 하이퍼 파라미터(hyper-parameter)등을 변경시켜 가며 더 좋은 분류 성능을 내는 알고리즘을 개발하였다. 또한 실험을 통해서 제안된 알고리즘이 농업현장에서 취득한 영상자료를 사용해서 오이를 등급별로 잘 구별하는 것을 

In [ ]:
# Install transformers if not already installed
!pip install transformers joblib

# Save the logistic regression classifier
import joblib
joblib.dump(lr_clf, 'domain_classifier.pkl')  # This saves the trained classifier

# Save tokenizer and model for offline use
from transformers import AutoTokenizer, AutoModel
model_name = "klue/bert-base"

# Save the tokenizer and model
tokenizer.save_pretrained("./klue_bert_tokenizer")
model.save_pretrained("./klue_bert_model")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
import joblib
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

# Load the classifier
lr_clf = joblib.load('./Model/domain_classifier.pkl')

# Load BERT tokenizer and model (assuming they're saved in these folders)
tokenizer = AutoTokenizer.from_pretrained('./Model/klue_bert_tokenizer')
model = AutoModel.from_pretrained('./Model/klue_bert_model')


In [ ]:
import torch
import numpy as np
import re
from nltk.tokenize import sent_tokenize
from statistics import mode

# Pre-loaded components (make sure you've already run these)
# lr_clf = joblib.load('domain_classifier.pkl')
# tokenizer = AutoTokenizer.from_pretrained('./klue_bert_tokenizer')
# model = AutoModel.from_pretrained('./klue_bert_model')

def honorifics(input):
    reFormal = r"(니다[.!]|니까[.?]*)"
    groupFormal = re.search(reFormal, input)
    reSemi = r"(요[.!?]*)"
    groupSemi = re.search(reSemi, input)
    if groupFormal:
        return 'Formal Honorific'
    elif groupSemi:
        return 'Casual Honorific'
    else:
        return 'Non-honorific'

def classify_input(text):
    # DOMAIN: Tokenize and get CLS embedding
    input_ids = torch.tensor([tokenizer.encode(text, add_special_tokens=True)[:512]])
    attention_mask = (input_ids != 0).long()
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
    cls_embedding = outputs[0][:, 0, :].numpy()
    prediction = lr_clf.predict(cls_embedding)[0]
    labels = ['Academic', 'General', 'Colloquial', 'Literary']
    domain_label = labels[prediction]

    # FORMALITY: Run sentence-wise honorific check
    sentences = sent_tokenize(text)
    styles = [honorifics(s) for s in sentences]
    formality = mode(styles) if styles else "Unknown"

    return {'Domain': domain_label, 'Formality': formality}


In [ ]:
sample = "유아교사의 핵심역량 8개 영역은 교사인성 및 전문성 개발, 놀이지원, 유아의 성장과 발달 등으로 구성되었다."
print("Predicted Domain:", classify_input(sample))

Predicted Domain: {'Domain': 'Academic', 'Formality': 'Non-honorific'}


In [ ]:
sample = "안녕하세요? 오늘은 유아교육의 중요성에 대해 이야기하겠습니다."
result = classify_input(sample)
print("Predicted Domain:", result['Domain'])
print("Formality:", result['Formality'])

Predicted Domain: Academic
Formality: Casual Honorific
